# 사규 본문 파싱 도구·포맷 비교 실험

대상은 1단 구조의 사규 본문이며, 장·절·조·항·호(목) 계층을 자동 집계합니다.

- 도구: `pdfplumber`, `Docling`, `pypdf`
- 포맷: PDF, HWPX
- HWPX는 위 세 도구의 직접 입력 대상으로 두지 않고 `변환 필요`로 기록합니다.
- 결과물: `parsing_results.csv`, `parsing_results.json`, 도구별 추출 텍스트, 도구별 계층 분석 JSON
- 별도 제공된 **파일별_본문_파싱_결과_양식.xlsx**에 CSV 결과를 붙여 넣고 수기 검수 점수를 입력합니다.


In [ ]:
# Colab 실행: 필요한 라이브러리 설치
!pip -q install "pypdf>=5.0" "pdfplumber>=0.11" "docling>=2.0"


## 1. 파일 업로드 및 폴더 구조

아래 셀에서 PDF/HWPX 파일을 업로드합니다. 파일명은 아래 `DOCUMENTS` 설정의 이름과 일치해야 합니다.

권장 파일명 예시:
- S01.pdf / S01.hwpx
- S02.pdf / S02.hwpx
- M01.pdf / M01.hwpx
- M02.pdf / M02.hwpx
- H01.pdf / H01.hwpx
- H02.pdf / H02.hwpx


In [ ]:
from google.colab import files
from pathlib import Path
import os, shutil

SOURCE_DIR = Path("/content/regulation_samples")
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for file_name in uploaded:
    shutil.move(file_name, SOURCE_DIR / file_name)

print("업로드 파일")
for path in sorted(SOURCE_DIR.iterdir()):
    print("-", path.name)


In [ ]:
# 시험 대상 6개 문서를 실제 파일명에 맞게 수정하세요.
# 파일명만 기입하면 됩니다. 경로는 SOURCE_DIR 기준으로 자동 처리합니다.
DOCUMENTS = [
    {
        "document_id": "S01",
        "difficulty": "쉬움",
        "feature": "1단 / 장·절·조·항·호(목) / 일반 문장 / 병합 없는 표",
        "pdf": "S01.pdf",
        "hwpx": "S01.hwpx",
    },
    {
        "document_id": "S02",
        "difficulty": "쉬움",
        "feature": "1단 / 장·절·조·항·호(목) / 일반 문장 / 병합 없는 표",
        "pdf": "S02.pdf",
        "hwpx": "S02.hwpx",
    },
    {
        "document_id": "M01",
        "difficulty": "보통",
        "feature": "1단 / 장·절·조·항·호(목) / 병합 표 포함",
        "pdf": "M01.pdf",
        "hwpx": "M01.hwpx",
    },
    {
        "document_id": "M02",
        "difficulty": "보통",
        "feature": "1단 / 장·절·조·항·호(목) / 병합 표 포함",
        "pdf": "M02.pdf",
        "hwpx": "M02.hwpx",
    },
    {
        "document_id": "H01",
        "difficulty": "어려움",
        "feature": "1단 / 장·절·조·항·호(목) / 병합 표 + 조직도·프로세스·복합 도형 포함",
        "pdf": "H01.pdf",
        "hwpx": "H01.hwpx",
    },
    {
        "document_id": "H02",
        "difficulty": "어려움",
        "feature": "1단 / 장·절·조·항·호(목) / 병합 표 + 조직도·프로세스·복합 도형 포함",
        "pdf": "H02.pdf",
        "hwpx": "H02.hwpx",
    },
]


## 2. 파서·공통 함수 정의

- PDF는 세 도구 모두 실행합니다.
- HWPX는 세 도구의 직접 입력 비교 대상이 아니므로 `변환 필요`로 기록합니다.
- Docling은 텍스트 기반 사규 PDF를 전제로 OCR을 끕니다. 스캔 PDF라면 `DOCLING_USE_OCR = True`로 바꾸고 OCR 엔진을 별도로 구성하세요.


In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError
import json
import re
import traceback

import pdfplumber
from pypdf import PdfReader

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

OUTPUT_DIR = Path("/content/parsing_outputs")
TEXT_DIR = OUTPUT_DIR / "text"
JSON_DIR = OUTPUT_DIR / "json"
OUTPUT_DIR.mkdir(exist_ok=True)
TEXT_DIR.mkdir(exist_ok=True)
JSON_DIR.mkdir(exist_ok=True)

TOOLS = ["pdfplumber", "Docling", "pypdf"]
DOCLING_USE_OCR = False  # 텍스트 PDF는 False 권장

def package_version(package_name: str) -> str:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "확인불가"

TOOL_VERSIONS = {
    "pdfplumber": package_version("pdfplumber"),
    "Docling": package_version("docling"),
    "pypdf": package_version("pypdf"),
}

def make_docling_converter() -> DocumentConverter:
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = DOCLING_USE_OCR
    pipeline_options.do_table_structure = True

    return DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

DOCLING_CONVERTER = make_docling_converter()

# 사규 본문의 핵심 구조를 자동 집계하는 정규식
HIERARCHY_PATTERNS = {
    "장": re.compile(r"(?m)^\s*제\s*\d+\s*장\b"),
    "절": re.compile(r"(?m)^\s*제\s*\d+\s*절\b"),
    "조": re.compile(r"(?m)^\s*제\s*\d+\s*조(?:\s*\([^\n)]*\))?"),
    "항": re.compile(r"(?m)^\s*(?:[①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮]|제\s*\d+\s*항\b)"),
    "호/목": re.compile(r"(?m)^\s*(?:\d+[.)]|[가-하][.)]|\([가-하]\))\s*"),
}

def analyse_hierarchy(text: str) -> dict:
    return {name: len(pattern.findall(text or "")) for name, pattern in HIERARCHY_PATTERNS.items()}

def get_pdf_page_count(file_path: Path) -> int:
    reader = PdfReader(str(file_path))
    return len(reader.pages)

def extract_with_pdfplumber(file_path: Path) -> tuple[str, int]:
    with pdfplumber.open(str(file_path)) as pdf:
        pages = len(pdf.pages)
        text = "\n".join(
            page.extract_text(x_tolerance=2, y_tolerance=2) or ""
            for page in pdf.pages
        )
    return text, pages

def extract_with_pypdf(file_path: Path) -> tuple[str, int]:
    reader = PdfReader(str(file_path))
    pages = len(reader.pages)
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    return text, pages

def extract_with_docling(file_path: Path) -> tuple[str, int]:
    result = DOCLING_CONVERTER.convert(str(file_path))
    document = result.document
    text = document.export_to_markdown()
    return text, get_pdf_page_count(file_path)

def run_single_test(document: dict, file_path: Path, parser_name: str) -> dict:
    suffix = file_path.suffix.lower()
    is_pdf = suffix == ".pdf"
    test_id = f"{document['document_id']}-{suffix.replace('.', '').upper()}-{parser_name}"

    base = {
        "시험 ID": test_id,
        "난이도": document["difficulty"],
        "동일문서 ID": document["document_id"],
        "파일명": file_path.name,
        "포맷": suffix.replace(".", "").upper(),
        "페이지 수": "",
        "문서 구조 특징": document["feature"],
        "대상 범위": "본문: 장·절·조·항·호(목)",
        "파싱도구": parser_name,
        "도구 버전": TOOL_VERSIONS.get(parser_name, ""),
        "직접입력 지원": "직접지원" if is_pdf else "변환 필요",
        "변환/사전처리": "없음" if is_pdf else "PDF 변환 후 실행",
        "실행 성공 여부": "",
        "오류/예외 요약": "",
        "처리 시간(초)": "",
        "추출 문자 수": "",
        "장 검출 수": "",
        "절 검출 수": "",
        "조 검출 수": "",
        "항 검출 수": "",
        "호/목 검출 수": "",
        "결과 파일/JSON 경로": "",
        "계층 정확도(0~100)": "",
        "읽기순서·누락(0~100)": "",
        "표·특수구조 보존(0~100)": "",
        "처리시간 점수": "",
        "종합점수": "",
        "종합판정": "",
        "검수자": "",
        "검수일": "",
        "비고": "",
    }

    # HWPX는 이 실험의 세 도구에서 직접입력 비교 대상이 아니므로 기록만 남깁니다.
    if not is_pdf:
        base["실행 성공 여부"] = "미지원"
        base["오류/예외 요약"] = "HWPX 직접입력 비교 제외: PDF 변환본으로 동일 도구 비교 수행"
        base["종합판정"] = "비교 제외/변환 필요"
        return base

    extractor_map = {
        "pdfplumber": extract_with_pdfplumber,
        "Docling": extract_with_docling,
        "pypdf": extract_with_pypdf,
    }
    extractor = extractor_map[parser_name]

    try:
        started = perf_counter()
        text, pages = extractor(file_path)
        elapsed = round(perf_counter() - started, 3)

        hierarchy = analyse_hierarchy(text)
        text_path = TEXT_DIR / f"{test_id}.txt"
        json_path = JSON_DIR / f"{test_id}.json"

        text_path.write_text(text, encoding="utf-8")
        json_path.write_text(
            json.dumps(
                {
                    "test_id": test_id,
                    "file_path": str(file_path),
                    "parser": parser_name,
                    "tool_version": TOOL_VERSIONS.get(parser_name, ""),
                    "page_count": pages,
                    "elapsed_seconds": elapsed,
                    "character_count": len(text),
                    "hierarchy_count": hierarchy,
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

        base.update({
            "페이지 수": pages,
            "실행 성공 여부": "성공",
            "처리 시간(초)": elapsed,
            "추출 문자 수": len(text),
            "장 검출 수": hierarchy["장"],
            "절 검출 수": hierarchy["절"],
            "조 검출 수": hierarchy["조"],
            "항 검출 수": hierarchy["항"],
            "호/목 검출 수": hierarchy["호/목"],
            "결과 파일/JSON 경로": str(json_path),
        })
    except Exception as exc:
        base.update({
            "실행 성공 여부": "실패",
            "오류/예외 요약": f"{type(exc).__name__}: {str(exc)[:500]}",
            "비고": traceback.format_exc(limit=1).strip(),
        })

    return base


In [ ]:
# 3. 전체 시험 실행
from collections import OrderedDict

results = []

for document in DOCUMENTS:
    file_map = {
        "PDF": SOURCE_DIR / document["pdf"],
        "HWPX": SOURCE_DIR / document["hwpx"],
    }

    for fmt, file_path in file_map.items():
        if not file_path.exists():
            print(f"[누락] {file_path.name} - 파일을 업로드했는지 DOCUMENTS 파일명을 확인하세요.")
            continue

        for tool in TOOLS:
            result = run_single_test(document, file_path, tool)
            results.append(result)
            print(
                f"[{result['실행 성공 여부']}] "
                f"{result['시험 ID']} | {result['처리 시간(초)']}초 | "
                f"조={result['조 검출 수']}"
            )

print(f"\n총 결과 행: {len(results)}")


In [ ]:
# 4. 결과 저장: Excel에서 바로 열리는 CSV + 상세 JSON
import csv

RESULT_COLUMNS = [
    "시험 ID", "난이도", "동일문서 ID", "파일명", "포맷", "페이지 수", "문서 구조 특징", "대상 범위",
    "파싱도구", "도구 버전", "직접입력 지원", "변환/사전처리",
    "실행 성공 여부", "오류/예외 요약", "처리 시간(초)", "추출 문자 수",
    "장 검출 수", "절 검출 수", "조 검출 수", "항 검출 수", "호/목 검출 수", "결과 파일/JSON 경로",
    "계층 정확도(0~100)", "읽기순서·누락(0~100)", "표·특수구조 보존(0~100)", "처리시간 점수",
    "종합점수", "종합판정", "검수자", "검수일", "비고",
]

csv_path = OUTPUT_DIR / "파일별_본문_파싱_결과_자동측정.csv"
json_path = OUTPUT_DIR / "파일별_본문_파싱_결과_전체.json"

with csv_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=RESULT_COLUMNS)
    writer.writeheader()
    writer.writerows(results)

json_path.write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("저장 완료")
print("-", csv_path)
print("-", json_path)

# CSV의 자동측정 열(A~V)을 제공된 엑셀 양식의 동일 열에 붙여 넣은 뒤,
# W~Y 수기 검수 점수를 입력하면 Z~AB의 점수·판정이 자동 계산됩니다.


In [ ]:
# 5. 다운로드
from google.colab import files

files.download(str(OUTPUT_DIR / "파일별_본문_파싱_결과_자동측정.csv"))
files.download(str(OUTPUT_DIR / "파일별_본문_파싱_결과_전체.json"))
